# 自由振动

系统在初始扰动（初始位移或初始速度）作用下，撤去外力后自发产生的振动称为**自由振动**。根据系统是否受到阻尼力的作用，自由振动可以分为无阻尼自由振动和有阻尼自由振动。

## 无阻尼自由振动

对于质量为 $m$、刚度为 $k$ 的单自由度系统，根据牛顿第二定律，其无阻尼自由振动的微分方程如公式 {eq}`eq:1-1` 所示：

$$
m\ddot{x} + kx = 0
$$ (eq:1-1)

将公式 {eq}`eq:1-1` 两边同除以 $m$，令 $\omega_n^2 = k/m$，方程可化为标准形式：

$$
\ddot{x} + \omega_n^2 x = 0
$$ (eq:1-2)

```{note}
**物理意义：** 式中的 $\omega_n$ 称为系统的**无阻尼圆频率（固有频率）**，单位为 rad/s。它完全由系统自身的固有属性（质量 $m$ 和刚度 $k$）决定，与外界初始条件无关。
```

该常系数线性齐次微分方程的通解为 $x(t) = A \cos(\omega_n t - \phi)$，表明系统将永远以固有频率 $\omega_n$ 进行等幅简谐振动。

## 有阻尼自由振动

在实际船舶工程中，结构不可避免地受到流体粘性或材料内摩擦的阻碍，这种耗散能量的作用称为**阻尼**。引入粘性阻尼 $c$ 后，系统的运动微分方程变为：

$$
m\ddot{x} + c\dot{x} + kx = 0
$$ (eq:1-3)

为了便于分析，我们引入阻尼比 $\zeta = c/(2\sqrt{mk})$，方程化为：

$$
\ddot{x} + 2\zeta\omega_n\dot{x} + \omega_n^2 x = 0
$$ (eq:1-4)

```{warning}
**阻尼比 $\zeta$ 的大小直接决定了系统的运动形态：**
1. $0 \le \zeta < 1$：**欠阻尼状态**。系统发生振幅按指数衰减的周期性振荡。
2. $\zeta = 1$：**临界阻尼状态**。系统不发生振荡，以最快速度回到平衡位置。
3. $\zeta > 1$：**过阻尼状态**。系统不发生振荡，极其缓慢地回到平衡位置。
```

 交互探索：阻尼如何影响振动？
 
 下面是一个基于单自由度系统精确解析解构建的仿真模型（设初始位移 $x_0=1$，初始速度 $v_0=0$，固有频率 $\omega_n=10$）。
 
 请尝试拖动下方的滑动条，观察阻尼比 $\zeta$ 从 0 增加到 1.5 时，振动曲线是如何从“等幅振荡”过渡到“迅速衰减”，最终变为“缓慢爬行”的过阻尼状态的。

In [1]:
import numpy as np
import json
from IPython.display import display, HTML

# 1. 基础物理参数设置
t = np.linspace(0, 10, 1200)  # 时间从 0 到 10 秒
omega_n = 10.0              # 固有频率
x0 = 1.0                    # 初始位移
zeta_init = 0.2

zeta_values = np.round(np.arange(0.0, 1.5001, 0.02), 2)
init_active = int(np.argmin(np.abs(zeta_values - zeta_init)))
t_values = np.round(t, 5).tolist()
tick_labels = [f"{value:.1f}" for value in np.arange(0.0, 1.51, 0.1)]
tick_values = np.round(np.arange(0.0, 1.51, 0.1), 1).tolist()
minor_tick_values = [float(value) for value in zeta_values if not np.isclose((value * 100) % 10, 0)]

html = f"""
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  #free-vibration-demo input[type="range"] {{
    -webkit-appearance: none;
    appearance: none;
    width: 100%;
    height: 6px;
    border-radius: 999px;
    background: rgba(148, 163, 184, 0.35);
    outline: none;
  }}

  #free-vibration-demo input[type="range"]::-webkit-slider-runnable-track {{
    height: 6px;
    border-radius: 999px;
    background: rgba(148, 163, 184, 0.35);
  }}

  #free-vibration-demo input[type="range"]::-webkit-slider-thumb {{
    -webkit-appearance: none;
    appearance: none;
    width: 18px;
    height: 18px;
    border-radius: 50%;
    background: #0f766e;
    border: 3px solid #ffffff;
    box-shadow: 0 4px 12px rgba(15, 118, 110, 0.28);
    margin-top: -6px;
    cursor: pointer;
  }}

  #free-vibration-demo input[type="range"]::-moz-range-track {{
    height: 6px;
    border: none;
    border-radius: 999px;
    background: rgba(148, 163, 184, 0.35);
  }}

  #free-vibration-demo input[type="range"]::-moz-range-progress {{
    height: 6px;
    border-radius: 999px;
    background: rgba(148, 163, 184, 0.35);
  }}

  #free-vibration-demo input[type="range"]::-moz-range-thumb {{
    width: 18px;
    height: 18px;
    border-radius: 50%;
    background: #0f766e;
    border: 3px solid #ffffff;
    box-shadow: 0 4px 12px rgba(15, 118, 110, 0.28);
    cursor: pointer;
  }}
</style>
<div id="free-vibration-demo" style="
    max-width: 1080px;
    margin: 1.2rem auto;
    padding: 1rem 1rem 0.8rem;
    border-radius: 22px;
    background: linear-gradient(145deg, #f7fffd 0%, #eef7ff 100%);
    box-shadow: 0 18px 45px rgba(15, 23, 42, 0.10);
    border: 1px solid rgba(148, 163, 184, 0.18);
    font-family: 'Segoe UI', 'PingFang SC', 'Microsoft YaHei', sans-serif;
">
  <div style="
      display: grid;
      grid-template-columns: repeat(4, minmax(0, 1fr));
      gap: 0.75rem;
      margin: 0 0 0.9rem;
  ">
    <div style="padding: 0.8rem 0.95rem; border-radius: 16px; background: rgba(255,255,255,0.78); border: 1px solid rgba(15,118,110,0.10);">
      <div style="font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.08em;">无阻尼</div>
      <div style="margin-top: 0.3rem; font-size: 14px; color: #0f172a;">持续等幅振荡，系统不耗能。</div>
    </div>
    <div style="padding: 0.8rem 0.95rem; border-radius: 16px; background: rgba(255,255,255,0.78); border: 1px solid rgba(15,118,110,0.10);">
      <div style="font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.08em;">欠阻尼</div>
      <div style="margin-top: 0.3rem; font-size: 14px; color: #0f172a;">仍会振荡，但振幅会随时间逐渐衰减。</div>
    </div>
    <div style="padding: 0.8rem 0.95rem; border-radius: 16px; background: rgba(255,255,255,0.78); border: 1px solid rgba(15,118,110,0.10);">
      <div style="font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.08em;">临界阻尼</div>
      <div style="margin-top: 0.3rem; font-size: 14px; color: #0f172a;">不振荡，并以最快速度回到平衡位置。</div>
    </div>
    <div style="padding: 0.8rem 0.95rem; border-radius: 16px; background: rgba(255,255,255,0.78); border: 1px solid rgba(15,118,110,0.10);">
      <div style="font-size: 12px; color: #64748b; text-transform: uppercase; letter-spacing: 0.08em;">过阻尼</div>
      <div style="margin-top: 0.3rem; font-size: 14px; color: #0f172a;">不振荡，但回位变慢，像“缓慢爬行”。</div>
    </div>
  </div>

  <div style="
      padding: 0.2rem 0.35rem 0.8rem;
  ">
    <div style="display:flex; justify-content:space-between; align-items:center; gap:0.8rem; margin-bottom:0.45rem; flex-wrap:wrap;">
      <label for="fv-zeta-slider" style="font-size: 14px; color: #334155; font-weight: 700;">
        当前阻尼比
        <span id="fv-zeta" style="margin-left:0.5rem; color:#0f766e;">ζ = 0.20</span>
        <span id="fv-state" style="margin-left:0.7rem; color:#0f172a;">系统状态：欠阻尼</span>
      </label>
    </div>
    <input id="fv-zeta-slider" type="range" min="0" max="{len(zeta_values) - 1}" step="1" value="{init_active}">
    <div style="position:relative; height:14px; margin-top:0.25rem;">
      <div style="position:absolute; left:9px; right:9px; top:0; bottom:0;">
        {"".join(
            f"<span style='position:absolute; left:{(value / 1.5) * 100:.4f}%; transform:translateX(-50%); top:4px; width:1px; height:6px; background:rgba(148,163,184,0.65);'></span>"
            for value in minor_tick_values
        )}
        {"".join(
            f"<span style='position:absolute; left:{(value / 1.5) * 100:.4f}%; transform:translateX(-50%); top:1px; width:1.6px; height:11px; background:rgba(100,116,139,0.86);'></span>"
            for value in tick_values
        )}
      </div>
    </div>
    <div style="position:relative; height:18px; margin-top:0.35rem; font-size:12px; color:#64748b;">
      <div style="position:absolute; left:9px; right:9px; top:0; bottom:0;">
        {"".join(
            (
                f"<span style='position:absolute; left:{(value / 1.5) * 100:.4f}%; "
                + ("transform:translateX(-50%);" if value not in (0.0, 1.5) else ("transform:none;" if value == 0.0 else "transform:translateX(-100%);"))
                + f" white-space:nowrap;'>{label}</span>"
            )
            for value, label in zip(tick_values, tick_labels)
        )}
      </div>
    </div>
  </div>

  <div style="
      display: grid;
      grid-template-columns: 1fr;
      gap: 1rem;
      align-items: stretch;
      margin-top: 0.2rem;
  ">
    <div style="
        padding: 0.8rem 0.95rem;
        border-radius: 18px;
        background: rgba(255,255,255,0.86);
        border: 1px solid rgba(148, 163, 184, 0.16);
    ">
      <div style="display:grid; grid-template-columns:minmax(360px, 1.75fr) minmax(220px, 0.72fr); gap:0.75rem; align-items:center;">
        <svg id="fv-oscillator" viewBox="0 0 360 172" style="width:100%; height:auto; display:block; border-radius:16px; background:linear-gradient(180deg, #ffffff 0%, #f7fbff 100%);">
          <rect x="0" y="0" width="360" height="172" rx="16" fill="transparent"></rect>
          <rect x="18" y="34" width="18" height="104" rx="6" fill="#94a3b8"></rect>
          <line x1="36" y1="68" x2="50" y2="68" stroke="rgba(15,23,42,0.22)" stroke-width="4" stroke-linecap="round"></line>
          <line x1="50" y1="56" x2="170" y2="56" stroke="#94a3b8" stroke-width="3" stroke-linecap="round"></line>
          <line x1="50" y1="80" x2="170" y2="80" stroke="#94a3b8" stroke-width="3" stroke-linecap="round"></line>
          <line x1="50" y1="56" x2="50" y2="80" stroke="#94a3b8" stroke-width="3" stroke-linecap="round"></line>
          <line id="fv-damper-piston" x1="92" y1="58" x2="92" y2="78" stroke="#64748b" stroke-width="4" stroke-linecap="round"></line>
          <line id="fv-damper-rod" x1="92" y1="68" x2="170" y2="68" stroke="#64748b" stroke-width="4" stroke-linecap="round"></line>
          <line x1="36" y1="108" x2="320" y2="108" stroke="rgba(148,163,184,0.35)" stroke-width="6" stroke-linecap="round"></line>
          <line id="fv-equilibrium" x1="258" y1="28" x2="258" y2="144" stroke="rgba(15,118,110,0.18)" stroke-width="2" stroke-dasharray="5 6"></line>
          <line id="fv-initial" x1="318" y1="28" x2="318" y2="144" stroke="rgba(249,115,22,0.45)" stroke-width="2" stroke-dasharray="4 5"></line>
          <polyline id="fv-spring" fill="none" stroke="#0f766e" stroke-width="4" stroke-linecap="round" stroke-linejoin="round"></polyline>
          <rect id="fv-mass" x="226" y="54" width="64" height="64" rx="14" fill="#14b8a6" stroke="#0f766e" stroke-width="3"></rect>
          <text x="20" y="24" fill="#64748b" style="font-size:12px;">固定端</text>
          <text x="295" y="22" fill="#f97316" style="font-size:12px;">初始位置</text>
          <text x="235" y="164" fill="#64748b" style="font-size:12px;">平衡位置</text>
        </svg>
        <div style="display:grid; grid-template-columns:minmax(0, 1fr) auto; gap:0.65rem; align-items:center;">
          <div style="display:grid; gap:0.7rem;">
            <div style="padding:0.65rem 0.72rem; border-radius:14px; background:#f8fffd; border:1px solid rgba(15,118,110,0.12);">
              <div style="font-size:12px; color:#64748b;">动画时间</div>
              <div id="fv-time" style="margin-top:0.2rem; font-size:16px; font-weight:700; color:#0f172a;">t = 0.00 s</div>
            </div>
            <div style="padding:0.65rem 0.72rem; border-radius:14px; background:#f8fffd; border:1px solid rgba(15,118,110,0.12);">
              <div style="font-size:12px; color:#64748b;">实时位移</div>
              <div id="fv-displacement" style="margin-top:0.2rem; font-size:16px; font-weight:700; color:#0f172a;">x = 1.000</div>
            </div>
          </div>
          <div style="display:flex; flex-direction:column; gap:0.55rem;">
            <button id="fv-play" style="padding: 0.72rem 1rem; border: none; border-radius: 999px; background: #0f766e; color: white; font-weight: 700; cursor: pointer;">播放</button>
            <button id="fv-reset" style="padding: 0.72rem 1rem; border: 1px solid rgba(15,118,110,0.18); border-radius: 999px; background: white; color: #0f172a; font-weight: 700; cursor: pointer;">重播</button>
          </div>
        </div>
      </div>
    </div>

    <div style="
        padding: 0.95rem;
        border-radius: 18px;
        background: rgba(255,255,255,0.86);
        border: 1px solid rgba(148, 163, 184, 0.16);
    ">
      <div id="fv-plot" style="width:100%; height:470px;"></div>
    </div>
  </div>
</div>

<script>
(function() {{
  const container = document.getElementById("free-vibration-demo");
  if (!container || container.dataset.initialized === "true") return;
  container.dataset.initialized = "true";

  const tValues = {json.dumps(t_values, ensure_ascii=False)};
  const zetaValues = {json.dumps(zeta_values.tolist(), ensure_ascii=False)};
  const omegaN = {omega_n};
  const x0 = {x0};
  const initActive = {init_active};
  const accent = "#0f766e";
  const playbackDuration = 8000;
  const equilibriumX = 258;
  const massWidth = 64;
  const motionScale = 60;
  const damperRodLength = 116;

  const stateEl = container.querySelector("#fv-state");
  const descEl = container.querySelector("#fv-desc");
  const zetaEl = container.querySelector("#fv-zeta");
  const timeEl = container.querySelector("#fv-time");
  const displacementEl = container.querySelector("#fv-displacement");
  const sliderEl = container.querySelector("#fv-zeta-slider");
  const playBtn = container.querySelector("#fv-play");
  const resetBtn = container.querySelector("#fv-reset");
  const plotEl = container.querySelector("#fv-plot");
  const springEl = container.querySelector("#fv-spring");
  const massEl = container.querySelector("#fv-mass");
  const damperPistonEl = container.querySelector("#fv-damper-piston");
  const damperRodEl = container.querySelector("#fv-damper-rod");

  let currentStep = initActive;
  let currentIndex = 0;
  let isPlaying = false;
  let startTimestamp = null;
  let rafId = null;
  let response = computeResponse(zetaValues[currentStep]);

  function computeResponse(zeta) {{
    const yMain = [];
    const yUp = [];
    const yDown = [];
    let state = "";
    let description = "";

    if (Math.abs(zeta) < 1e-12) {{
      state = "无阻尼";
      description = "等幅简谐振荡";
      for (const time of tValues) {{
        const value = x0 * Math.cos(omegaN * time);
        yMain.push(value);
        yUp.push(x0);
        yDown.push(-x0);
      }}
    }} else if (zeta < 1.0) {{
      state = "欠阻尼";
      description = "振荡且振幅指数衰减";
      const omegaD = omegaN * Math.sqrt(1 - zeta * zeta);
      for (const time of tValues) {{
        const decay = Math.exp(-zeta * omegaN * time);
        const value = decay * (
          x0 * Math.cos(omegaD * time) +
          (zeta * omegaN * x0 / omegaD) * Math.sin(omegaD * time)
        );
        const envelope = x0 * decay;
        yMain.push(value);
        yUp.push(envelope);
        yDown.push(-envelope);
      }}
    }} else if (Math.abs(zeta - 1.0) < 1e-12) {{
      state = "临界阻尼";
      description = "不振荡且最快回到平衡";
      for (const time of tValues) {{
        const value = (x0 + omegaN * x0 * time) * Math.exp(-omegaN * time);
        yMain.push(value);
        yUp.push(null);
        yDown.push(null);
      }}
    }} else {{
      state = "过阻尼";
      description = "不振荡但回到平衡更慢";
      const root = Math.sqrt(zeta * zeta - 1);
      const s1 = -omegaN * (zeta - root);
      const s2 = -omegaN * (zeta + root);
      const c1 = (s2 * x0) / (s2 - s1);
      const c2 = (-s1 * x0) / (s2 - s1);
      for (const time of tValues) {{
        const value = c1 * Math.exp(s1 * time) + c2 * Math.exp(s2 * time);
        yMain.push(value);
        yUp.push(null);
        yDown.push(null);
      }}
    }}

    return {{ zeta, state, description, yMain, yUp, yDown }};
  }}

  function buildPlotData(index) {{
    const partialX = tValues.slice(0, index + 1);
    const partialY = response.yMain.slice(0, index + 1);
    const markerX = [tValues[index]];
    const markerY = [response.yMain[index]];

    return [
      {{
        x: tValues,
        y: response.yDown,
        mode: "lines",
        line: {{ color: "rgba(15, 118, 110, 0.42)", width: 1.2, dash: "dot" }},
        hoverinfo: "skip",
        showlegend: false,
      }},
      {{
        x: tValues,
        y: response.yUp,
        mode: "lines",
        fill: "tonexty",
        fillcolor: "rgba(15, 118, 110, 0.18)",
        line: {{ color: "rgba(15, 118, 110, 0.42)", width: 1.2, dash: "dot" }},
        name: "衰减包络",
        hoverinfo: "skip",
      }},
      {{
        x: tValues,
        y: response.yMain,
        mode: "lines",
        name: "完整响应",
        line: {{ color: "rgba(15, 23, 42, 0.18)", width: 2 }},
        hovertemplate: "t = %{{x:.2f}} s<br>x = %{{y:.3f}}<extra></extra>",
      }},
      {{
        x: partialX,
        y: partialY,
        mode: "lines",
        name: "已绘响应",
        line: {{ color: accent, width: 3.2 }},
        hovertemplate: "t = %{{x:.2f}} s<br>x = %{{y:.3f}}<extra></extra>",
      }},
      {{
        x: markerX,
        y: markerY,
        mode: "markers",
        name: "当前点",
        marker: {{ size: 10, color: accent, line: {{ color: "white", width: 2 }} }},
        hovertemplate: "t = %{{x:.2f}} s<br>x = %{{y:.3f}}<extra></extra>",
      }}
    ];
  }}

  function buildLayout() {{
    return {{
      margin: {{ l: 62, r: 18, t: 18, b: 52 }},
      paper_bgcolor: "rgba(0,0,0,0)",
      plot_bgcolor: "#ffffff",
      xaxis: {{
        title: "时间 t (s)",
        range: [0, 10],
        showgrid: true,
        gridcolor: "rgba(148, 163, 184, 0.22)",
        zeroline: false,
        showline: true,
        linecolor: "rgba(15, 23, 42, 0.28)",
        ticks: "outside",
      }},
      yaxis: {{
        title: "归一化位移 x",
        range: [-1.2, 1.2],
        showgrid: true,
        gridcolor: "rgba(148, 163, 184, 0.22)",
        zeroline: true,
        zerolinecolor: "rgba(15, 23, 42, 0.18)",
        zerolinewidth: 1.5,
      }},
      legend: {{
        orientation: "h",
        yanchor: "bottom",
        y: 1.02,
        xanchor: "right",
        x: 0.98,
        bgcolor: "rgba(255,255,255,0.72)",
        bordercolor: "rgba(148, 163, 184, 0.20)",
        borderwidth: 1,
      }},
      hoverlabel: {{
        bgcolor: "#ffffff",
        bordercolor: "rgba(15, 118, 110, 0.30)",
        font: {{ color: "#0f172a" }},
      }},
    }};
  }}

  function updateReadouts(index) {{
    stateEl.textContent = `系统状态：${{response.state}}`;
    if (descEl) descEl.textContent = response.description;
    zetaEl.textContent = `ζ = ${{response.zeta.toFixed(2)}}`;
    timeEl.textContent = `t = ${{tValues[index].toFixed(2)}} s`;
    displacementEl.textContent = `x = ${{response.yMain[index].toFixed(3)}}`;
  }}

  function buildSpringPoints(startX, endX, y) {{
    const points = [];
    const straight = 16;
    const turns = 9;
    const amplitude = 18;
    points.push(`${{startX}},${{y}}`);
    points.push(`${{startX + straight}},${{y}}`);
    const usable = Math.max(endX - startX - straight * 2, 30);
    const step = usable / (turns * 2);
    for (let i = 0; i < turns * 2; i += 1) {{
      const x = startX + straight + step * (i + 1);
      const offset = i % 2 === 0 ? -amplitude : amplitude;
      points.push(`${{x}},${{y + offset}}`);
    }}
    points.push(`${{endX - straight}},${{y}}`);
    points.push(`${{endX}},${{y}}`);
    return points.join(" ");
  }}

  function updateOscillator(index) {{
    const displacement = response.yMain[index];
    const massX = equilibriumX + displacement * motionScale;
    const massLeft = massX - massWidth / 2;
    const damperAnchorX = massLeft;
    const pistonX = damperAnchorX - damperRodLength;
    massEl.setAttribute("x", massLeft.toFixed(2));
    springEl.setAttribute("points", buildSpringPoints(36, massLeft + 2, 108));
    damperPistonEl.setAttribute("x1", pistonX.toFixed(2));
    damperPistonEl.setAttribute("x2", pistonX.toFixed(2));
    damperPistonEl.setAttribute("y1", "58");
    damperPistonEl.setAttribute("y2", "78");
    damperRodEl.setAttribute("x1", pistonX.toFixed(2));
    damperRodEl.setAttribute("x2", damperAnchorX.toFixed(2));
    damperRodEl.setAttribute("y1", "68");
    damperRodEl.setAttribute("y2", "68");
  }}

  function renderPlot(index, fullRedraw) {{
    const partialX = tValues.slice(0, index + 1);
    const partialY = response.yMain.slice(0, index + 1);
    const markerX = [tValues[index]];
    const markerY = [response.yMain[index]];

    if (fullRedraw) {{
      Plotly.react(plotEl, buildPlotData(index), buildLayout(), {{
        displayModeBar: false,
        responsive: true,
      }});
    }} else {{
      Plotly.restyle(plotEl, {{ x: [partialX], y: [partialY] }}, [3]);
      Plotly.restyle(plotEl, {{ x: [markerX], y: [markerY] }}, [4]);
    }}
  }}

  function syncScene(index, fullRedraw = false) {{
    currentIndex = index;
    updateReadouts(index);
    updateOscillator(index);
    renderPlot(index, fullRedraw);
  }}

  function stopAnimation() {{
    isPlaying = false;
    startTimestamp = null;
    if (rafId) cancelAnimationFrame(rafId);
    rafId = null;
    playBtn.textContent = "播放";
  }}

  function stepAnimation(timestamp) {{
    if (!isPlaying) return;
    if (startTimestamp === null) {{
      startTimestamp = timestamp - (currentIndex / (tValues.length - 1)) * playbackDuration;
    }}
    const progress = Math.min((timestamp - startTimestamp) / playbackDuration, 1);
    const nextIndex = Math.min(Math.floor(progress * (tValues.length - 1)), tValues.length - 1);
    if (nextIndex !== currentIndex) {{
      syncScene(nextIndex, false);
    }}
    if (progress >= 1) {{
      stopAnimation();
      return;
    }}
    rafId = requestAnimationFrame(stepAnimation);
  }}

  function startAnimation(fromBeginning = false) {{
    if (fromBeginning) {{
      stopAnimation();
      syncScene(0, true);
    }}
    isPlaying = true;
    startTimestamp = null;
    playBtn.textContent = "暂停";
    rafId = requestAnimationFrame(stepAnimation);
  }}

  function toggleAnimation() {{
    if (isPlaying) {{
      stopAnimation();
    }} else {{
      if (currentIndex >= tValues.length - 1) {{
        syncScene(0, true);
      }}
      startAnimation(false);
    }}
  }}

  function applyZeta(stepIndex) {{
    currentStep = stepIndex;
    sliderEl.value = String(stepIndex);
    response = computeResponse(zetaValues[stepIndex]);
    stopAnimation();
    syncScene(0, true);
    startAnimation(false);
  }}

  sliderEl.addEventListener("input", (event) => {{
    applyZeta(Number(event.target.value));
  }});

  playBtn.addEventListener("click", () => {{
    toggleAnimation();
  }});

  resetBtn.addEventListener("click", () => {{
    applyZeta(currentStep);
  }});

  syncScene(0, true);
  startAnimation(false);
}})();
</script>
"""

display(HTML(html))